# Titanic Survival — Reproducible Data Workflow

- **Name:** Vu Tran
- **Dataset:** Titanic - Machine Learning from Disaster (Kaggle `train.csv`) — https://www.kaggle.com/c/titanic/data

**Project description:** This notebook builds a reproducible data workflow for the Titanic passenger dataset: it loads the raw CSV, cleans it with documented functions, explores survival patterns, and visualizes the results. The goal is to identify which passenger characteristics, such as sex, class, and age, relate to survival. The cleaned dataset will serve as a foundation for later machine learning projects.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("train.csv")
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Data Quality Check

In [3]:
print(df.info())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None

Duplicate rows: 0

Missing values per column:
Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             

## Data Cleaning

The raw data has inconsistent column names and missing values in `Age` (177), `Cabin` (687) and `Embarked` (2).
Each issue is handled by a documented function. The raw DataFrame `df` is left unchanged.

In [4]:
def standardize_column_names(df):
    """Return a copy of df with column names converted to lowercase snake_case.

    Example: 'PassengerId' -> 'passenger_id', 'SibSp' -> 'sib_sp'.
    """
    new_cols = (df.columns
                .str.replace(r"(?<=[a-z])(?=[A-Z])", "_", regex=True)
                .str.lower())
    return df.set_axis(new_cols, axis=1).copy()


def impute_age(df):
    """Fill missing ages with the median age of passengers with the same sex and class.

    Adds an 'age_was_missing' flag BEFORE filling, so imputed ages can be
    separated from observed ages in later analysis.
    """
    out = df.copy()
    out["age_was_missing"] = out["age"].isna()
    group_median = out.groupby(["sex", "pclass"])["age"].transform("median")
    out["age"] = out["age"].fillna(group_median)
    return out


def handle_cabin_and_embarked(df):
    """Replace the sparse 'cabin' column with a 'has_cabin' flag and fill 'embarked'.

    Cabin is ~77% missing, so it is reduced to True/False and dropped.
    Embarked has 2 missing values, which are filled with the most common port.
    """
    out = df.copy()
    out["has_cabin"] = out["cabin"].notna()
    out["embarked"] = out["embarked"].fillna(out["embarked"].mode()[0])
    return out.drop(columns="cabin")


def clean_titanic(df):
    """Run the full cleaning pipeline and return the cleaned DataFrame."""
    return (df.pipe(standardize_column_names)
              .pipe(impute_age)
              .pipe(handle_cabin_and_embarked))

In [5]:
df_clean = clean_titanic(df)

assert len(df_clean) == len(df), "Cleaning must not drop rows"
assert df_clean[["age", "embarked"]].notna().all().all(), "Missing values remain"

print(df_clean.shape)
print(df_clean.isna().sum())
df_clean.head()

(891, 13)
passenger_id       0
survived           0
pclass             0
name               0
sex                0
age                0
sib_sp             0
parch              0
ticket             0
fare               0
embarked           0
age_was_missing    0
has_cabin          0
dtype: int64


,passenger_id,survived,pclass,name,sex,age,sib_sp,parch,ticket,fare,embarked,age_was_missing,has_cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,False,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,False,True
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,False,True
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,False,False


## Exploratory Data Analysis

Reusable functions for summary statistics, survival rates by group, and correlation with survival.

In [6]:
def summarize_numeric(df, cols):
    """Return summary statistics (count, mean, std, min, quartiles, max, skew) for numeric columns."""
    summary = df[cols].describe().T
    summary["skew"] = df[cols].skew()
    return summary


def survival_rate_by(df, group_cols, target="survived"):
    """Return passenger count, survivors and survival rate for each group in group_cols."""
    return (df.groupby(group_cols, observed=True)[target]
              .agg(passengers="count", survivors="sum", survival_rate="mean")
              .round(3))


def correlation_with_target(df, target="survived"):
    """Return the Pearson correlation of each numeric column with the target, strongest first.

    Sex is encoded as is_female (1/0) so it can be included. The ID column is excluded.
    Correlation measures linear association only, not cause.
    """
    numeric = (df.select_dtypes(include=["number", "bool"])
                 .drop(columns="passenger_id", errors="ignore")
                 .astype(float))
    numeric["is_female"] = (df["sex"] == "female").astype(float)
    corr = numeric.corr()[target].drop(target)
    return corr.reindex(corr.abs().sort_values(ascending=False).index).round(3)

In [7]:
print(f"Overall survival rate: {df_clean['survived'].mean():.1%}")
summarize_numeric(df_clean, ["age", "fare", "sib_sp", "parch"])

Overall survival rate: 38.4%


,count,mean,std,min,25%,50%,75%,max,skew
age,891.0,29.112424,13.304424,0.42,21.5000,26.0000,36.0,80.0000,0.534083
fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292,4.787317
sib_sp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000,3.695352
parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000,2.749117


In [8]:
survival_rate_by(df_clean, "sex")

,passengers,survivors,survival_rate
sex,,,
female,314,233,0.742
male,577,109,0.189


In [9]:
survival_rate_by(df_clean, "pclass")

,passengers,survivors,survival_rate
pclass,,,
1,216,136,0.630
2,184,87,0.473
3,491,119,0.242


In [10]:
survival_rate_by(df_clean, ["pclass", "sex"])

passengers  survivors  survival_rate
pclass sex                                         
1      female          94         91          0.968
       male           122         45          0.369
2      female          76         70          0.921
       male           108         17          0.157
3      female         144         72          0.500
       male           347         47          0.135

In [11]:
# Is missing age itself related to survival? (needed for the bias section of the report)
survival_rate_by(df_clean, "age_was_missing")

,passengers,survivors,survival_rate
age_was_missing,,,
False,714,290,0.406
True,177,52,0.294


In [12]:
correlation_with_target(df_clean)

is_female          0.543
pclass            -0.338
has_cabin          0.317
fare               0.257
age_was_missing   -0.092
parch              0.082
age               -0.060
sib_sp            -0.035
Name: survived, dtype: float64